In [2]:
import os
import pandas as pd
from sklearn.metrics import cohen_kappa_score

def calculate_excel_kappa(file_path, col1_name, col2_name):
    """
    Reads an Excel sheet, handles missing data for the specified columns, 
    and calculates linear and quadratic weighted Kappa.
    """
    if not os.path.exists(file_path):
        print(f"❌ Error: The file '{file_path}' could not be found. Please check the path.")
        return
        
    try:
        # Load the Excel file
        df = pd.read_excel(file_path)
        
        # Verify columns exist
        if col1_name not in df.columns or col2_name not in df.columns:
            print(f"❌ Error: One or both columns missing from Excel sheet.")
            print(f"Available columns: {list(df.columns)}")
            return
            
        # Clean missing values via pairwise deletion
        df_clean = df[[col1_name, col2_name]].dropna()
        
        if len(df_clean) == 0:
            print("❌ Error: No complete pairs of data left after removing missing values (NaNs).")
            return
            
        # Ensure data is parsed as integers for ordinal calculation
        rater1 = df_clean[col1_name].astype(int)
        rater2 = df_clean[col2_name].astype(int)
        
        # Calculate Kappa metrics
        quadratic_kappa = cohen_kappa_score(rater1, rater2, weights='quadratic')
        linear_kappa = cohen_kappa_score(rater1, rater2, weights='linear')
        unweighted_kappa = cohen_kappa_score(rater1, rater2, weights=None)
        
        # Output Report
        print(f"📊 Reliability Report for Tumor-Infiltrating Lymphocytes (TILs):")
        print(f"File: {os.path.basename(file_path)}")
        print(f"Evaluated Pairs: {len(df_clean)} patients")
        print("-" * 60)
        print(f"Quadratic Weighted Kappa : {quadratic_kappa:.3f}")
        print(f"Linear Weighted Kappa    : {linear_kappa:.3f}")
        print(f"Unweighted Cohen's Kappa : {unweighted_kappa:.3f}")
        
    except Exception as e:
        print(f"❌ An error occurred during processing: {str(e)}")

# --- CONFIGURATION ---
# Change this path to point to your actual Excel file location
EXCEL_FILE = "/Users/Francisco/Downloads/BRC#683_path inter-reader cases - JAG_SCW.xlsx" 

# Using your exact column names
READER_JAG = "TLS/resection_jag"
READER_SCW = "TLS/resection_scw"

calculate_excel_kappa(EXCEL_FILE, READER_JAG, READER_SCW)

📊 Reliability Report for Tumor-Infiltrating Lymphocytes (TILs):
File: BRC#683_path inter-reader cases - JAG_SCW.xlsx
Evaluated Pairs: 30 patients
------------------------------------------------------------
Quadratic Weighted Kappa : 0.870
Linear Weighted Kappa    : 0.829
Unweighted Cohen's Kappa : 0.763


In [6]:
import os
import pandas as pd
import pingouin as pg

def calculate_excel_icc(file_path, col1_name, col2_name, id_col=None):
    """
    Reads an Excel sheet, reshapes the data to long format,
    and computes all forms of Intraclass Correlation Coefficients (ICC).
    """
    if not os.path.exists(file_path):
        print(f"❌ Error: The file '{file_path}' could not be found.")
        return
        
    try:
        # 1. Load the Excel file
        df = pd.read_excel(file_path)
        
        # Create a temporary ID column if none is provided
        if id_col is None or id_col not in df.columns:
            df['Patient_ID'] = df.index
            id_col = 'Patient_ID'
            
        # Verify rating columns exist
        if col1_name not in df.columns or col2_name not in df.columns:
            print(f"❌ Error: One or both rating columns missing from Excel sheet.")
            return
            
        # 2. Clean missing values via pairwise deletion
        df_clean = df[[id_col, col1_name, col2_name]].dropna()
        
        print(f"📊 Running ICC Analysis for Tumor-Infiltrating Lymphocytes (TILs):")
        print(f"Evaluated Cohort: {len(df_clean)} patients")
        print("-" * 80)
        
        # 3. Reshape from WIDE to LONG format
        df_long = pd.melt(
            df_clean, 
            id_vars=[id_col], 
            value_vars=[col1_name, col2_name],
            var_name='Rater', 
            value_name='Score'
        )
        
        # 4. Compute ICC
        icc_table = pg.intraclass_corr(
            data=df_long, 
            targets=id_col, 
            raters='Rater', 
            ratings='Score'
        )
        
        # 5. Print the FULL table safely (No column index errors)
        print(icc_table.to_string(index=False))
        
    except Exception as e:
        print(f"❌ An error occurred during processing: {str(e)}")

# --- CONFIGURATION ---
EXCEL_FILE = "/Users/Francisco/Downloads/BRC#683_path inter-reader cases - JAG_SCW.xlsx" 
READER_JAG = "TLS/resection_jag"
READER_SCW = "TLS/resection_scw"
PATIENT_ID_COLUMN = None # Set to your ID column name if you have one

calculate_excel_icc(EXCEL_FILE, READER_JAG, READER_SCW, id_col=PATIENT_ID_COLUMN)

📊 Running ICC Analysis for Tumor-Infiltrating Lymphocytes (TILs):
Evaluated Cohort: 30 patients
--------------------------------------------------------------------------------
 Type             Description      ICC         F  df1  df2         pval        CI95%
 ICC1  Single raters absolute 0.873599 14.822660   29   30 3.815803e-11 [0.75, 0.94]
 ICC2    Single random raters 0.873678 14.970149   29   29 6.231040e-11 [0.75, 0.94]
 ICC3     Single fixed raters 0.874766 14.970149   29   29 6.231040e-11 [0.75, 0.94]
ICC1k Average raters absolute 0.932536 14.822660   29   30 3.815803e-11 [0.86, 0.97]
ICC2k   Average random raters 0.932581 14.970149   29   29 6.231040e-11 [0.86, 0.97]
ICC3k    Average fixed raters 0.933200 14.970149   29   29 6.231040e-11 [0.86, 0.97]
